In [0]:
from pyspark.sql.types import *
import json
from pyspark.sql.functions import explode, col

In [0]:
root = '/Volumes/s3_data_platform/s3_data/raw/lol/'
table = 'matches'
path = f'{root}{table}/'
checkpoint_loc = f'{root}{table}_checkpont_loc/'
table_path = f'bronze.lol.{table}'

In [0]:
with open(f"./schemas/{table}.json", "r") as f:
    schema = StructType.fromJson(json.load(f))

In [0]:
df = spark.readStream.format("json").schema(schema).load(path)
df = df.select(
    *[col(f"info.{c}").alias(f"info_{c}") for c in df.select("info.*").columns],
    *[col(f"metadata.{c}").alias(f"metadata_{c}") for c in df.select("metadata.*").columns]
)

In [0]:
stream = (df.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_loc)
            .trigger(availableNow=True)
            .toTable(table_path)
)
stream.awaitTermination()